In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

notebook_path = Path().absolute()
sys.path.append(str(notebook_path.parent))

In [3]:
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from neural_controllers import NeuralController
import utils

In [4]:
model_type = 'llama'

if model_type=='llama':
    model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

    language_model = AutoModelForCausalLM.from_pretrained(
        model_id, device_map="cuda"
    )

    use_fast_tokenizer = "LlamaForCausalLM" not in language_model.config.architectures
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=use_fast_tokenizer, padding_side="left", legacy=False)
    model_name='llama_3_8b_it'
    assistant_tag = '<|start_header_id|>assistant<|end_header_id|>'
    
elif model_type=='gemma':

    tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-9b-it")
    language_model = AutoModelForCausalLM.from_pretrained(
        "google/gemma-2-9b-it",
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )
    model_name='gemma_2_9b_it'
    
tokenizer.pad_token_id = 0 if tokenizer.pad_token_id is None else tokenizer.pad_token_id

2025-02-21 17:19:03.822829: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1740179944.035038 1945483 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1740179944.108237 1945483 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-21 17:19:04.687433: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [5]:
data_dir = "../data/poetry"

dataset = utils.poetry_dataset(data_dir=data_dir, tokenizer=tokenizer, assistant_tag=assistant_tag)

train 200 test 0
train 200 test 0


In [6]:
concept_types = ['prose', 'poetry']

controllers = {}
for concept_type in tqdm(concept_types):
    
    other_type = [k for k in concept_types if k != concept_type][0]
    
    train_data = dataset[concept_type]['train']
    test_data = dataset[concept_type]['test']
        
    controller = NeuralController(
        language_model,
        tokenizer,
        rfm_iters=8,
        batch_size=2,
        control_method='rfm'
    )
    
    controller.compute_directions(train_data['inputs'], train_data['labels'])
    
    controllers[concept_type] = controller
    

  0%|          | 0/2 [00:00<?, ?it/s]

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 2
M_batch_size         : 2048
n_components         : 5

use_concat False
Getting activations from forward passes



100%|██████████| 100/100 [00:14<00:00,  6.75it/s]

  0%|          | 0/31 [00:00<?, ?it/s]

train X shape: torch.Size([160, 4096]) train y shape: torch.Size([160, 1]) val X shape: torch.Size([40, 4096]) val y shape: torch.Size([40, 1])
Fit time: 0.06858086585998535
Score time: 0.011106014251708984
Update best time: 0.02802300453186035
Update M time: 0.02403736114501953
Fit time: 0.0012149810791015625
Score time: 0.0005803108215332031
Update best time: 0.00037741661071777344
Update M time: 0.005140066146850586
Fit time: 0.0020673274993896484
Score time: 0.0004885196685791016
Update best time: 0.0003769397735595703
Update M time: 0.0050868988037109375
Fit time: 0.0021767616271972656
Score time: 0.0004470348358154297
Update best time: 0.00039577484130859375
Update M time: 0.0050623416900634766
Fit time: 0.00217437744140625
Score time: 0.0004286766052246094
Update best time: 0.024761676788330078
Update M time: 0.005273103713989258
Fit time: 0.002154827117919922
Score time: 0.00045490264892578125
Update best time: 0.024646997451782227
Update M time: 0.005255699157714844
Fit time: 


  3%|▎         | 1/31 [00:05<02:31,  5.06s/it]

...


 97%|█████████▋| 30/31 [02:18<00:04,  4.47s/it]

train X shape: torch.Size([160, 4096]) train y shape: torch.Size([160, 1]) val X shape: torch.Size([40, 4096]) val y shape: torch.Size([40, 1])
Fit time: 0.0011582374572753906
Score time: 0.00038123130798339844
Update best time: 0.018100738525390625
Update M time: 0.007459402084350586
Fit time: 0.001990795135498047
Score time: 0.0004982948303222656
Update best time: 0.01843881607055664
Update M time: 0.005423069000244141
Fit time: 0.0022003650665283203
Score time: 0.00042557716369628906
Update best time: 0.0003154277801513672
Update M time: 0.005151510238647461
Fit time: 0.0022721290588378906
Score time: 0.00041365623474121094
Update best time: 0.00032210350036621094
Update M time: 0.005147695541381836
Fit time: 0.0022783279418945312
Score time: 0.0004124641418457031
Update best time: 0.0003292560577392578
Update M time: 0.005137920379638672
Fit time: 0.002274036407470703
Score time: 0.00040411949157714844
Update best time: 0.00033473968505859375
Update M time: 0.00515294075012207
Fit 


100%|██████████| 31/31 [02:23<00:00,  4.62s/it]


tensors torch.Size([200, 4096]) direction torch.Size([4096])
tensors torch.Size([200, 4096]) direction torch.Size([4096])
tensors torch.Size([200, 4096]) direction torch.Size([4096])
tensors torch.Size([200, 4096]) direction torch.Size([4096])
tensors torch.Size([200, 4096]) direction torch.Size([4096])
tensors torch.Size([200, 4096]) direction torch.Size([4096])
tensors torch.Size([200, 4096]) direction torch.Size([4096])
tensors torch.Size([200, 4096]) direction torch.Size([4096])
tensors torch.Size([200, 4096]) direction torch.Size([4096])
tensors torch.Size([200, 4096]) direction torch.Size([4096])
tensors torch.Size([200, 4096]) direction torch.Size([4096])
tensors torch.Size([200, 4096]) direction torch.Size([4096])
tensors torch.Size([200, 4096]) direction torch.Size([4096])
tensors torch.Size([200, 4096]) direction torch.Size([4096])
tensors torch.Size([200, 4096]) direction torch.Size([4096])
tensors torch.Size([200, 4096]) direction torch.Size([4096])
tensors torch.Size([200,


100%|██████████| 2/2 [04:59<00:00, 149.88s/it]


In [8]:
for concept_type in concept_types:
    controller = controllers[concept_type]    
    controller.save(concept=f'{concept_type}', model_name=model_name, path='../directions/')

# Control

In [ ]:
concept_types = ['prose', 'poetry']

controllers = {}
for concept_type in concept_types:
    
    controller = NeuralController(
        language_model,
        tokenizer,
        control_method='rfm'
    )
    
    other_type = [k for k in concept_types if k!=concept_type][0]
    
    controller.load(concept=f'{concept_type}', model_name=model_name, path='../directions/')
    
    controllers[concept_type] = controller
    

In [ ]:
concept_type = "prose"
# concept_type = "poetry"
controller = controllers[concept_type]

raw_inputs = [
    # f"How should I treat a cold?",
    f"What can I buy in a grocery store?",
    # f"What might a student study in school?",
    # f"Tell me about something interesting.",
    # f"Give me advice for applying to jobs.",
]
inputs = [controller.format_prompt(x) for x in raw_inputs]

num_new_tokens = 150

coef=0.4 #llama 
# coef=9

layers = list(range(-1, -31, -1))
# layers = list(range(-1, -41, -1))

gens=[]
print()
for i in inputs:
    print("Prompt:", i)
    print("===== No Control =====")
    print(controller.generate(i, max_new_tokens=num_new_tokens, do_sample=False).replace(i, ""))
    print()
    
    print(f"===== + {concept_type} Control =====")
    gen = controller.generate(i, layers_to_control=layers, control_coef=coef, 
                                max_new_tokens=num_new_tokens, do_sample=False).replace(i, "")
    gens.append(gen)
    print(gen)
    print()
    print()